# import models

In [1]:
import ete3
from ete3 import Tree
import numpy as np
import pandas as pd
import string
from random import choice, choices
import os
from pathlib import Path

In [2]:
import os
import math
import numpy as np
import pandas as pd

from ete3 import Tree
from joblib import Parallel, delayed

def get_dist_to_ancestor(tree, ancestor_node, target_node):
    dist = 0
    current = target_node
    while True:
        if current == ancestor_node:
            dist += 1
            break
        if current.up is None:
            # 已到根仍未找到
            break
        current = current.up
        dist += 1
    return dist

def compute_subtree_depth_info(tree):
    nodes = [i for i in tree.traverse() if not i.is_leaf()]
    ## Dr (distance to root)
    ## Ds (distance to farthest leaf)
    # mdepth = ((Dr/Dt) + (1 - Ds/Dt))/2 
    # node.height = 1 - mdepth
    Dr_vals = {}
    Ds_vals = {}
    for node in nodes:
        # for Dr calculate
        up_dist = 0
        tmp_up = node
        while tmp_up.up is not None:
            tmp_up = tmp_up.up
            up_dist += 1
        Dr_vals[node.name] = float(up_dist)
        # for Ds calculate
        farthest_leaf = node.get_farthest_leaf()
        down_dist = 0
        tmp_down = farthest_leaf[0]
        while tmp_down != node and tmp_down is not None:
            tmp_down = tmp_down.up
            down_dist += 1
        Ds_vals[node.name] = float(down_dist)


    # Dt (max Ds)
    Dt = max(Ds_vals.values())
    # calculate node.height
    #  mdepth = ((Dr/Dt) + (1 - Ds/Dt))/2 
    #  node.height = 1 - mdepth
    node_depth = {}
    node_height = {}
    for re_node in nodes:
        Dr = Dr_vals[re_node.name]
        Ds = Ds_vals[re_node.name]
        node_depth[re_node.name] =  Ds / Dt
        mdepth = ((Dr / Dt) + (1 - Ds / Dt)) / 2
        node_height[re_node.name] = 1 - mdepth
        

    # dict
    subtree_depth = {
        'Dr': Dr_vals,
        'Ds': Ds_vals,
        'depth': node_depth,
        'height': node_height,
        'max_ds': Dt,
    }
    return subtree_depth

# generate the two phase transition matrix

In [3]:
phase_1_transmx = pd.read_csv("/mnt/data5/disk/yangwj/Scripts/Fig2/new_two_phase_tree/phase_1_transmatrix.txt", sep='\t', index_col=0)
phase_2_transmx = pd.read_csv("/mnt/data5/disk/yangwj/Scripts/Fig2/new_two_phase_tree/phase_2_transmatrix_2.txt", sep='\t', index_col=0)

In [4]:
phase_1_transmx

,A,B,C,D
A,0.64,0.12,0.12,0.12
B,0.12,0.64,0.12,0.12
C,0.12,0.12,0.64,0.12
D,0.12,0.12,0.12,0.64


In [5]:
phase_2_transmx

,A,B,C,D,E,F,G,H
A,1,0,0.0,0.0,0.00,0.00,0.00,0.00
B,0,1,0.0,0.0,0.00,0.00,0.00,0.00
C,0,0,0.7,0.1,0.10,0.10,0.00,0.00
D,0,0,0.1,0.7,0.10,0.10,0.00,0.00
E,0,0,0.0,0.0,0.64,0.12,0.12,0.12
F,0,0,0.0,0.0,0.12,0.64,0.12,0.12
G,0,0,0.0,0.0,0.12,0.12,0.64,0.12
H,0,0,0.0,0.0,0.12,0.12,0.12,0.64


# new generation of tree

In [6]:
phase_tree_bb = Tree("/mnt/data5/disk/yangwj/Scripts/Fig2/new_two_phase_tree/new_two_phase_tree_50000_format1.nwk", format=1)

In [7]:
phase_tree_2 = phase_tree_bb.copy()

In [8]:
# 2. name each node and cite depth and mheight in each node
i = 1
for each_node in phase_tree_2.traverse():
    each_node.name = "N_" + str(i)
    each_node.dist = 1
    i += 1

In [9]:
subtree_depth_infos_2 = compute_subtree_depth_info(phase_tree_2)

In [10]:
subtree_depth_infos_2

{'Dr': {'N_1': 0.0,
  'N_2': 1.0,
  'N_3': 1.0,
  'N_4': 2.0,
  'N_5': 2.0,
  'N_6': 2.0,
  'N_7': 2.0,
  'N_8': 3.0,
  'N_9': 3.0,
  'N_10': 3.0,
  'N_11': 3.0,
  'N_12': 3.0,
  'N_13': 3.0,
  'N_14': 3.0,
  'N_15': 3.0,
  'N_16': 4.0,
  'N_17': 4.0,
  'N_18': 4.0,
  'N_19': 4.0,
  'N_20': 4.0,
  'N_21': 4.0,
  'N_22': 4.0,
  'N_23': 4.0,
  'N_24': 4.0,
  'N_25': 4.0,
  'N_26': 4.0,
  'N_27': 4.0,
  'N_28': 4.0,
  'N_29': 4.0,
  'N_30': 4.0,
  'N_31': 4.0,
  'N_32': 5.0,
  'N_33': 5.0,
  'N_34': 5.0,
  'N_35': 5.0,
  'N_36': 5.0,
  'N_37': 5.0,
  'N_38': 5.0,
  'N_39': 5.0,
  'N_40': 5.0,
  'N_41': 5.0,
  'N_42': 5.0,
  'N_43': 5.0,
  'N_44': 5.0,
  'N_45': 5.0,
  'N_46': 5.0,
  'N_47': 5.0,
  'N_48': 5.0,
  'N_49': 5.0,
  'N_50': 5.0,
  'N_51': 5.0,
  'N_52': 5.0,
  'N_53': 5.0,
  'N_54': 5.0,
  'N_55': 5.0,
  'N_56': 5.0,
  'N_57': 5.0,
  'N_58': 5.0,
  'N_59': 5.0,
  'N_60': 5.0,
  'N_61': 5.0,
  'N_62': 5.0,
  'N_63': 5.0,
  'N_64': 6.0,
  'N_65': 6.0,
  'N_66': 6.0,
  'N_67': 6.0

In [11]:
## generate tree infos with cell state
for e_n in phase_tree_2.traverse("preorder"):
    if e_n.is_root():
        e_n.add_features(state = "A")
    else:
        if not e_n.is_leaf():
            #use_mheight = subtree_depth_infos_2["height"][e_n.name]
            use_mheight = subtree_depth_infos_2["depth"][e_n.name]
        else:
            #use_mheight = subtree_depth_infos_2["height"][e_n.up.name]
            use_mheight = subtree_depth_infos_2["depth"][e_n.up.name]
        if use_mheight > 0.7:
            use_transmx = phase_1_transmx
        else:
            use_transmx = phase_2_transmx
        use_states = use_transmx.columns.to_list()
        ancestor_n_state = e_n.up.state
        #print(e_n.up.state)
        #print(use_transmx)
        e_n.add_features(state = choices(use_states, weights=use_transmx.loc[ancestor_n_state])[0])

In [17]:
from collections import defaultdict

## =========== define functions ==============
def write_results(tree_obj, node_infos_path, name = False):
    ## add name to tree
    node_state_infos = defaultdict(list)
    i = 1
    for each_node in tree_obj.traverse():
        if not name:
            each_node.name = "N_" + str(i)
        each_node.dist = 1
        i += 1
        ##
        node_state_infos["NodeName"].append(each_node.name)
        node_state_infos["state"].append(each_node.state)
        node_state_infos["is_leaf"].append(each_node.is_leaf())
        node_state_infos["is_root"].append(each_node.is_root())
    tree_info_df = pd.DataFrame.from_dict(node_state_infos)
    ## write out results
    #tree_obj.write(outfile=tree_path, format = 1)
    tree_info_df.to_csv(node_infos_path, sep = '\t', index = False)

In [19]:
save_path = "/mnt/data5/disk/yangwj/Scripts/Fig2/new_two_phase_tree/"
write_results(tree_obj = phase_tree_2,
              node_infos_path = os.path.join(save_path, "new_two_phase_tree_50000_depth_nodeInfo.txt"),
              name = False)

In [15]:
## check state composition
check_state_infos = defaultdict(list)
j = 1
for each_n in phase_tree_2.traverse():
#     if each_n.name == "":
#         each_n.name = "N_" + str(j)
#     each_n.dist = 1
#     j += 1
    check_state_infos["NodeName"].append(each_n.name)
    check_state_infos["state"].append(each_n.state)
    check_state_infos["is_leaf"].append(each_n.is_leaf())
    check_state_infos["is_root"].append(each_n.is_root())
    #check_state_infos["state"].append(each_n.state)

In [17]:
len(check_state_infos['NodeName'])

99999

In [18]:
len(check_state_infos['state'])

99999

In [19]:
check_state_infos_df = pd.DataFrame.from_dict(check_state_infos)

In [20]:
check_state_infos_df

,NodeName,state,is_leaf,is_root
0,N_1,A,False,True
1,N_2,B,False,False
2,N_3,A,False,False
3,N_4,B,False,False
4,N_5,C,False,False
...,...,...,...,...
99994,N_99995,F,True,False
99995,N_99996,A,True,False
99996,N_99997,A,True,False
99997,N_99998,A,True,False


In [21]:
leaf_state_infos_df = check_state_infos_df[check_state_infos_df['is_leaf'] == True]

In [22]:
leaf_state_infos_df['state'].value_counts()

B    13595
A     9221
F     6739
E     6620
H     6122
G     5841
D      945
C      917
Name: state, dtype: int64